# Actor-Critric on CartPole-v1

Actor-Critic Algorithm is a type of reinforcement learning algorithm that combines two parts i.e the Actor which selects actions and the Critic which evaluates them. This helps the agent learn more effectively by balancing decision-making and feedback. In the actor-critic method the actor learns how to make decisions and the critic checks how good those decisions are. This dual role helps the agent explore new actions while also using what it has learned and make the learning process better and more balanced.

Let's understand how the Actor-Critic algorithm works in practice. Below is an implementation of a simple Actor-Critic algorithm using TensorFlow and OpenAI Gym to train an agent in the CartPole environment.

## Step 1: Import Libraries

In [21]:
import numpy as np
import tensorflow as tf
import gym

## Step 2: Creating CartPole Environment

Create the CartPole environment using the gym.make() function from the Gym library because it provides a standardized and convenient way to interact with various reinforcement learning tasks.

In [22]:
# Create the CartPole Environment
env = gym.make('CartPole-v1', new_step_api=True)

## Step 3: Defining Actor and Critic Networks

Actor and the Critic are implemented as neural networks using TensorFlow's Keras API.
Actor network maps the state to a probability distribution over actions.
Critic network estimates the state's value.

In [23]:
# Define the actor and critic networks
actor = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(env.action_space.n, activation='softmax')
])

critic = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

## Step 4: Defining Optimizers and Loss Functions
We use Adam optimizer for both networks.

In [24]:
# Define optimizer and loss functions
actor_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
critic_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

## Step 5: Training Loop
The training loop runs for 1000 episodes with the agent interacting with the environment, calculating advantages and updating both the actor and critic.

In [27]:
# Main training loop
num_episodes = 1000
gamma = 0.99

for episode in range(num_episodes):
    state, _ = env.reset(return_info=True)
    episode_reward = 0

    with tf.GradientTape(persistent=True) as tape:
        for t in range(1, 10000):  # Limit the number of time steps
            # Choose an action using the actor
            action_probs = actor(np.array([state]))
            action = np.random.choice(env.action_space.n, p=action_probs.numpy()[0])

            # Take the chosen action and observe the next state and reward
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Compute the advantage
            state_value = critic(np.array([state]))[0, 0]
            next_state_value = critic(np.array([next_state]))[0, 0]
            advantage = reward + gamma * next_state_value - state_value

            # Compute actor and critic losses
            actor_loss = -tf.math.log(action_probs[0, action]) * advantage
            critic_loss = tf.square(advantage)

            episode_reward += reward

            # Update actor and critic
            actor_gradients = tape.gradient(actor_loss, actor.trainable_variables)
            critic_gradients = tape.gradient(critic_loss, critic.trainable_variables)
            actor_optimizer.apply_gradients(zip(actor_gradients, actor.trainable_variables))
            critic_optimizer.apply_gradients(zip(critic_gradients, critic.trainable_variables))

            if done:
                break
            state = next_state # Update the state for the next iteration

    if episode % 10 == 0:
        print(f'Episode {episode}, Reward: {episode_reward}')

env.close()

Episode 0, Reward: 9.0
Episode 10, Reward: 20.0
Episode 20, Reward: 27.0
Episode 30, Reward: 60.0
Episode 40, Reward: 14.0
Episode 50, Reward: 36.0
Episode 60, Reward: 38.0
Episode 70, Reward: 94.0
Episode 80, Reward: 40.0
Episode 90, Reward: 84.0
Episode 100, Reward: 100.0


KeyboardInterrupt: 